# Auditoria de diversidade angular e energia do banco VQE
## Versão para apresentação — gráficos autoexplicativos

Esta versão foi ajustada para que **cada figura carregue a própria mensagem principal**. Os gráficos apresentam valores-chave, porcentagens, linhas de referência, identificação dos picos, destaque da solução exata e uma conclusão curta dentro da própria imagem.

## Como usar

1. Coloque este notebook na mesma pasta de `01_generate_dataset.ipynb`.
2. O banco esperado está em `notebooks/results/merged.pkl`.
3. Reinicie o kernel e execute **Run All**.
4. As figuras serão exportadas em 300 dpi para `outputs_auditoria_vqe/figures`.

A versão procura explicitamente `merged.pkl` e evita carregar `problem_bundle.pkl`, que contém objetos simbólicos do Qiskit.

In [ ]:
# Caso seja necessário:
# %pip install numpy pandas matplotlib scipy scikit-learn pyarrow

from pathlib import Path
import ast
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore", category=FutureWarning)

plt.rcParams.update({
    "figure.figsize": (13, 7),
    "figure.dpi": 120,
    "axes.titlesize": 16,
    "axes.labelsize": 12,
    "font.size": 11,
})

RANDOM_STATE = 42


def info_box(ax, lines, location=(0.985, 0.97), fontsize=10):
    text = "\n".join(str(line) for line in lines)
    ax.text(
        location[0], location[1], text,
        transform=ax.transAxes,
        ha="right", va="top", fontsize=fontsize,
        bbox={
            "boxstyle": "round,pad=0.45",
            "facecolor": "white",
            "edgecolor": "0.45",
            "alpha": 0.92,
        },
        zorder=10,
    )


def label_horizontal_bars(ax, bars, labels):
    for bar, label in zip(bars, labels):
        ax.annotate(
            label,
            xy=(bar.get_width(), bar.get_y() + bar.get_height()/2),
            xytext=(6, 0), textcoords="offset points",
            va="center", fontsize=9,
        )


def short_takeaway(ax, text):
    ax.text(
        0.5, -0.17, text,
        transform=ax.transAxes,
        ha="center", va="top",
        fontsize=10, fontweight="bold",
    )

# 1. Configuração

O erro energético é calculado por

\[
\Delta E_r=\left|E_r-E_{\mathrm{exato}}\right|.
\]

Para este experimento:

\[
E_{\mathrm{exato}}\approx-0.8181062577496281,
\qquad
x^\star=\texttt{1001011000}.
\]

A tolerância `ENERGY_BIN_TOL` serve somente para agrupar energias próximas. Ela não prova degenerescência física.

In [ ]:
# =========================
# CONFIGURAÇÃO PRINCIPAL
# =========================

# Caminho relativo esperado quando este notebook está na pasta "notebooks".
DATA_PATH = Path("results/merged.pkl")

# Colunas reais do merged.pkl produzido pelo notebook de geração.
THETA_VECTOR_COLUMN = "best_parameters"
ENERGY_COLUMN = "objective_function_value"
BITSTRING_COLUMN = "most_frequent_bitstring"

N_QUBITS = 10
EXACT_ENERGY = -0.8181062577496281
EXACT_BITSTRING = "1001011000"

TOP_BITSTRINGS = 10
TOP_EQUIVALENCE_CLASSES = 15
ENERGY_BIN_TOL = 1e-3
K_RANGE = range(2, 9)
SILHOUETTE_SAMPLE_SIZE = 3000

UNIMODAL_EXAMPLE_INDEX = 1
BIMODAL_EXAMPLE_INDEX = 11

OUTPUT_DIR = Path("outputs_auditoria_vqe")
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("Figuras:", FIGURE_DIR.resolve())

# 2. Carregamento e padronização

O notebook cria três objetos:

- `THETA`: matriz \(N\times p\);
- `ENERGY`: energia final de cada execução;
- `BITSTRING`: bitstring dominante.

Os ângulos são envolvidos em

\[
[-\pi,\pi)
\]

por

\[
\theta_{\mathrm{wrap}}=(\theta+\pi)\bmod 2\pi-\pi.
\]

In [ ]:
SUPPORTED_SUFFIXES = {".csv", ".parquet", ".pq", ".pkl", ".pickle", ".json", ".npz", ".npy"}


def load_any(path):
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    if suffix in {".pkl", ".pickle"}:
        return pd.read_pickle(path)
    if suffix == ".json":
        try:
            return pd.read_json(path)
        except ValueError:
            return json.loads(path.read_text(encoding="utf-8"))
    if suffix == ".npz":
        obj = np.load(path, allow_pickle=True)
        return {key: obj[key] for key in obj.files}
    if suffix == ".npy":
        return np.load(path, allow_pickle=True)

    raise ValueError(f"Formato não suportado: {suffix}")


def resolve_configured_path(configured_path):
    """Localiza merged.pkl sem depender do diretório de execução do Jupyter."""
    configured_path = Path(configured_path)

    candidates = [
        configured_path,
        Path.cwd() / configured_path,
        Path.cwd() / "results" / "merged.pkl",
        Path.cwd() / "notebooks" / "results" / "merged.pkl",
        Path.cwd().parent / "results" / "merged.pkl",
        Path.cwd().parent / "notebooks" / "results" / "merged.pkl",
    ]

    # Remove duplicatas mantendo a ordem.
    unique_candidates = []
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate not in unique_candidates:
            unique_candidates.append(candidate)

    for candidate in unique_candidates:
        if candidate.is_file():
            return candidate.resolve()

    attempted = "\n".join(f"  - {candidate}" for candidate in unique_candidates)
    raise FileNotFoundError(
        "Não encontrei results/merged.pkl. Caminhos testados:\n"
        f"{attempted}\n\n"
        "Confirme que este notebook está dentro da pasta notebooks ou ajuste DATA_PATH."
    )


def candidate_score(path):
    """Usado apenas como fallback; evita arquivos de configuração do Qiskit."""
    name = path.name.lower()

    if any(token in name for token in [
        "problem_bundle", "bundle", "progress", "config", "manifest"
    ]):
        return -10_000

    weights = {
        "merged": 100,
        "vqe_dataset_combined": 90,
        "combined": 40,
        "part_": 20,
        "theta": 8,
        "teta": 8,
        "vqe": 6,
        "dataset": 5,
        "result": 3,
        "energy": 3,
        "10000": 2,
    }
    return sum(weight for token, weight in weights.items() if token in name)


def discover_file():
    """Fallback seguro caso DATA_PATH seja colocado como None."""
    preferred = [
        Path.cwd() / "results" / "merged.pkl",
        Path.cwd() / "notebooks" / "results" / "merged.pkl",
        Path.cwd().parent / "results" / "merged.pkl",
        Path.cwd().parent / "notebooks" / "results" / "merged.pkl",
        Path.cwd() / "results" / "vqe_dataset_combined.pkl",
    ]
    for path in preferred:
        if path.is_file():
            return path.resolve()

    candidates = []
    for root in [Path.cwd(), Path.cwd().parent]:
        if not root.exists():
            continue
        for path in root.rglob("*"):
            if (
                path.is_file()
                and path.suffix.lower() in SUPPORTED_SUFFIXES
                and candidate_score(path) > -10_000
            ):
                candidates.append(path)

    candidates = list(dict.fromkeys(candidates))
    candidates.sort(
        key=lambda p: (candidate_score(p), p.stat().st_mtime),
        reverse=True,
    )

    if not candidates:
        raise FileNotFoundError(
            "Nenhum banco compatível foi encontrado. Defina DATA_PATH explicitamente."
        )

    print("Arquivo selecionado pelo fallback:", candidates[0])
    return candidates[0].resolve()


def is_safe_column_sequence(value):
    """Evita chamar np.asarray em objetos do Qiskit com __array__ simbólico."""
    return isinstance(value, (list, tuple, np.ndarray, pd.Series, pd.Index))


def as_dataframe(obj):
    if isinstance(obj, pd.DataFrame):
        return obj.copy()

    if isinstance(obj, pd.Series):
        return obj.to_frame()

    if isinstance(obj, dict):
        # Alguns pickles guardam o DataFrame dentro de uma chave.
        for key in ["dataframe", "df", "dataset", "data", "results", "records"]:
            if key in obj:
                nested = obj[key]
                if isinstance(nested, (pd.DataFrame, pd.Series, list, tuple, np.ndarray, dict)):
                    try:
                        return as_dataframe(nested)
                    except (TypeError, ValueError):
                        pass

        # Converte apenas sequências comuns. Não tenta converter circuitos/gates Qiskit.
        safe_arrays = {
            key: value
            for key, value in obj.items()
            if is_safe_column_sequence(value)
        }

        if safe_arrays:
            lengths = []
            valid_arrays = {}
            for key, value in safe_arrays.items():
                try:
                    length = len(value)
                except TypeError:
                    continue
                valid_arrays[key] = value
                lengths.append(length)

            if valid_arrays and len(set(lengths)) == 1:
                return pd.DataFrame(valid_arrays)

        # Uma lista de registros dentro do dicionário.
        for value in obj.values():
            if isinstance(value, list) and value and isinstance(value[0], dict):
                return pd.DataFrame(value)

        raise TypeError(
            "O pickle carregado é um dicionário, mas não contém o banco tabular esperado. "
            "Provavelmente foi selecionado problem_bundle.pkl em vez de results/merged.pkl."
        )

    if isinstance(obj, (list, tuple)):
        if len(obj) and isinstance(obj[0], dict):
            return pd.DataFrame(list(obj))
        return pd.DataFrame(obj)

    if isinstance(obj, np.ndarray):
        if obj.dtype.names:
            return pd.DataFrame(obj)
        if obj.ndim == 2:
            return pd.DataFrame(obj)
        if obj.ndim == 1 and len(obj) and isinstance(obj[0], dict):
            return pd.DataFrame(list(obj))

    raise TypeError(
        f"Objeto do tipo {type(obj).__name__} não pôde ser convertido para DataFrame."
    )


def parse_vector(value):
    if isinstance(value, np.ndarray):
        return value.astype(float).ravel()
    if isinstance(value, (list, tuple)):
        return np.asarray(value, dtype=float).ravel()
    if isinstance(value, str):
        text = value.strip()
        text = re.sub(r"^array\((.*)\)$", r"\1", text)
        text = re.sub(r"np\.float\d*\(([^)]+)\)", r"\1", text)
        try:
            return np.asarray(ast.literal_eval(text), dtype=float).ravel()
        except Exception:
            numbers = re.findall(
                r"[-+]?(?:\d*\.\d+|\d+\.?)(?:[eE][-+]?\d+)?",
                text,
            )
            if numbers:
                return np.asarray([float(x) for x in numbers], dtype=float)
    raise ValueError(f"Vetor não interpretado: {type(value)}")


def to_real(value):
    if value is None:
        return np.nan
    if isinstance(value, (float, int, complex, np.number)):
        return float(np.real(value))
    text = re.sub(
        r"np\.float\d*\(([^)]+)\)",
        r"\1",
        str(value).strip(),
    )
    try:
        return float(complex(text.replace(" ", "")).real)
    except Exception:
        match = re.search(
            r"[-+]?(?:\d*\.\d+|\d+\.?)(?:[eE][-+]?\d+)?",
            text,
        )
        return float(match.group()) if match else np.nan


def clean_bitstring(value, width=None):
    if value is None:
        return None
    groups = re.findall(r"[01]+", str(value))
    if not groups:
        return None
    bit = max(groups, key=len)
    return bit.zfill(width) if width else bit


def detect_column(df, manual, candidates):
    if manual is not None:
        if manual not in df.columns:
            raise KeyError(
                f"Coluna configurada não encontrada: {manual}. "
                f"Colunas disponíveis: {list(df.columns)}"
            )
        return manual

    lower = {str(column).lower(): column for column in df.columns}
    for candidate in candidates:
        if candidate.lower() in lower:
            return lower[candidate.lower()]
    for column in df.columns:
        if any(
            candidate.lower() in str(column).lower()
            for candidate in candidates
        ):
            return column
    return None


def detect_theta(df):
    if THETA_VECTOR_COLUMN is not None:
        if THETA_VECTOR_COLUMN not in df.columns:
            raise KeyError(
                f"Coluna theta '{THETA_VECTOR_COLUMN}' não encontrada. "
                f"Colunas disponíveis: {list(df.columns)}"
            )
        vectors = [parse_vector(value) for value in df[THETA_VECTOR_COLUMN]]
        lengths = {len(vector) for vector in vectors}
        if len(lengths) != 1:
            raise ValueError(
                f"Vetores theta com comprimentos diferentes: {sorted(lengths)}"
            )
        return np.vstack(vectors)

    indexed = []
    for column in df.columns:
        normalized = str(column).lower().replace("θ", "theta")
        match = re.search(r"(?:theta|teta)[_\[\s-]*(\d+)", normalized)
        if match:
            indexed.append((int(match.group(1)), column))

    if indexed:
        indexed.sort(key=lambda item: item[0])
        columns = [column for _, column in indexed]
        return df[columns].apply(
            lambda series: series.map(to_real)
        ).to_numpy(dtype=float)

    candidates = [
        column for column in df.columns
        if any(
            token in str(column).lower()
            for token in [
                "theta", "teta", "optimal_point", "parameters", "params"
            ]
        )
    ]
    for column in candidates:
        try:
            vectors = [parse_vector(value) for value in df[column]]
            if len({len(vector) for vector in vectors}) == 1 and len(vectors[0]) > 1:
                return np.vstack(vectors)
        except Exception:
            continue

    raise KeyError(
        "Theta não encontrado. Informe THETA_VECTOR_COLUMN ou use colunas theta_0, theta_1, ..."
    )


selected_path = (
    resolve_configured_path(DATA_PATH)
    if DATA_PATH is not None
    else discover_file()
)

loaded_object = load_any(selected_path)
raw_df = as_dataframe(loaded_object)

print("Arquivo correto carregado:", selected_path)
print("Tipo carregado:", type(loaded_object).__name__)
print("Dimensão bruta:", raw_df.shape)
print("Colunas disponíveis:", list(raw_df.columns))

required_columns = {
    THETA_VECTOR_COLUMN,
    ENERGY_COLUMN,
    BITSTRING_COLUMN,
}
missing_columns = required_columns.difference(raw_df.columns)
if missing_columns:
    raise KeyError(
        f"O arquivo foi aberto, mas faltam as colunas: {sorted(missing_columns)}"
    )

print("\nValidação do banco concluída: merged.pkl possui as colunas necessárias.")

In [ ]:
theta_matrix = detect_theta(raw_df)

energy_col = detect_column(
    raw_df, ENERGY_COLUMN,
    ["objective_function_value", "energy", "vqe_energy", "final_energy",
     "optimal_value", "objective_value", "fun", "eigenvalue"]
)
if energy_col is None:
    raise KeyError("Coluna de energia não encontrada. Defina ENERGY_COLUMN.")

bit_col = detect_column(
    raw_df, BITSTRING_COLUMN,
    ["most_frequent_bitstring", "dominant_bitstring", "best_bitstring",
     "most_probable_bitstring", "solution_bitstring", "bitstring"]
)

energy_values = raw_df[energy_col].map(to_real).to_numpy(dtype=float)

if bit_col is None:
    bitstrings = np.array([None] * len(raw_df), dtype=object)
    print("[AVISO] Bitstring não encontrado. Os gráficos correspondentes serão ignorados.")
else:
    bitstrings = raw_df[bit_col].map(
        lambda value: clean_bitstring(value, N_QUBITS)
    ).to_numpy(dtype=object)

valid = np.all(np.isfinite(theta_matrix), axis=1) & np.isfinite(energy_values)

THETA = theta_matrix[valid]
ENERGY = energy_values[valid]
BITSTRING = bitstrings[valid]
THETA_WRAPPED = (THETA + np.pi) % (2 * np.pi) - np.pi

analysis_df = pd.DataFrame({
    "row_id": np.flatnonzero(valid),
    "energy": ENERGY,
    "bitstring": BITSTRING,
})
analysis_df["absolute_energy_error"] = np.abs(ENERGY - EXACT_ENERGY)
analysis_df["relative_energy_error"] = analysis_df["absolute_energy_error"] / abs(EXACT_ENERGY)

print(f"Execuções válidas: {len(analysis_df):,}")
print(f"Parâmetros: {THETA_WRAPPED.shape[1]}")
print("Energia:", energy_col)
print("Bitstring:", bit_col)

try:
    analysis_df.to_parquet(TABLE_DIR / "analysis_dataset_standardized.parquet", index=False)
except Exception:
    analysis_df.to_csv(TABLE_DIR / "analysis_dataset_standardized.csv", index=False)

## Conferência inicial

**Pergunta respondida:** o banco carregado é o esperado?

Confira o número de execuções, o número de parâmetros e as frações de erro antes de apresentar os gráficos.

In [ ]:
summary = pd.Series({
    "n_execucoes": len(analysis_df),
    "n_parametros": THETA_WRAPPED.shape[1],
    "energia_exata": EXACT_ENERGY,
    "energia_media": analysis_df["energy"].mean(),
    "erro_medio": analysis_df["absolute_energy_error"].mean(),
    "erro_mediano": analysis_df["absolute_energy_error"].median(),
    "fracao_erro_menor_1e-3": (analysis_df["absolute_energy_error"] < 1e-3).mean(),
    "fracao_erro_menor_1e-2": (analysis_df["absolute_energy_error"] < 1e-2).mean(),
    "fracao_bitstring_exato": (analysis_df["bitstring"] == EXACT_BITSTRING).mean(),
})
display(summary.to_frame("valor"))
summary.to_csv(TABLE_DIR / "summary_global.csv", header=["valor"])

# 3. Variância circular de cada componente

### Pergunta respondida

**Quais parâmetros ficam concentrados e quais variam realmente no círculo?**

\[
C_j=\frac1N\sum_r\cos\theta_j^{(r)},
\qquad
S_j=\frac1N\sum_r\sin\theta_j^{(r)},
\]

\[
R_j=\sqrt{C_j^2+S_j^2},
\qquad
V_j=1-R_j.
\]

- \(V_j\approx0\): componente concentrada;
- \(V_j\) maior: componente angularmente mais dispersa.

Como \(V_j=1-R_j\), a concentração e a variância carregam a mesma informação em sentidos opostos.

In [ ]:
C = np.mean(np.cos(THETA_WRAPPED), axis=0)
S = np.mean(np.sin(THETA_WRAPPED), axis=0)
RESULTANT_LENGTH = np.sqrt(C**2 + S**2)
CIRCULAR_VARIANCE = 1 - RESULTANT_LENGTH
CIRCULAR_MEAN = np.arctan2(S, C)

circular_stats = pd.DataFrame({
    "theta_index": np.arange(THETA_WRAPPED.shape[1]),
    "circular_mean": CIRCULAR_MEAN,
    "resultant_length_R": RESULTANT_LENGTH,
    "circular_variance": CIRCULAR_VARIANCE,
    "linear_std": np.std(THETA_WRAPPED, axis=0),
})
circular_stats.to_csv(TABLE_DIR / "circular_statistics.csv", index=False)
display(circular_stats)

mean_v = float(CIRCULAR_VARIANCE.mean())
max_idx = int(np.argmax(CIRCULAR_VARIANCE))
min_idx = int(np.argmin(CIRCULAR_VARIANCE))
max_v = float(CIRCULAR_VARIANCE[max_idx])
min_v = float(CIRCULAR_VARIANCE[min_idx])

fig, ax = plt.subplots()
ax.bar(circular_stats["theta_index"], circular_stats["circular_variance"])
ax.axhline(mean_v, linestyle="--", linewidth=1.4, label=f"Média global = {mean_v:.3f}")

ax.annotate(
    f"Mais variável\nθ{max_idx}: V={max_v:.3f}",
    xy=(max_idx, max_v), xytext=(max_idx + 2.0, max_v + 0.035),
    arrowprops={"arrowstyle": "->"}, fontsize=10,
)
ax.annotate(
    f"Mais estável\nθ{min_idx}: V={min_v:.4f}",
    xy=(min_idx, min_v), xytext=(max(0, min_idx - 8), max(0.055, min_v + 0.055)),
    arrowprops={"arrowstyle": "->"}, fontsize=10,
)

ax.set_title("Quais componentes de θ realmente variam no círculo?")
ax.set_xlabel("Índice do parâmetro")
ax.set_ylabel("Variância circular  V = 1 − R")
ax.set_xticks(np.arange(THETA_WRAPPED.shape[1]))
ax.grid(axis="y", alpha=0.25)
ax.legend(loc="upper center")
info_box(ax, [
    f"N = {len(THETA_WRAPPED):,} execuções",
    f"V média = {mean_v:.3f}",
    f"Maior dispersão: θ{max_idx}",
    f"Maior concentração: θ{min_idx}",
])
short_takeaway(ax, "Barras altas indicam liberdade angular; barras próximas de zero indicam parâmetros quase fixos.")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "01_variancia_circular_INFORMATIVO.png", dpi=300, bbox_inches="tight")
plt.show()

# 4. Multimodalidade angular

### Pergunta respondida

**Cada componente termina em uma única região angular ou em mais de uma?**

O procedimento constrói um histograma circular, suaviza-o respeitando a união entre \(-\pi\) e \(+\pi\) e conta picos suficientemente proeminentes.

- 1 modo: uma região predominante;
- 2 modos: duas regiões predominantes.

Isso descreve \(p(\theta_j)\) isoladamente. Não é o número de clusters globais nem de mínimos físicos.

In [ ]:
def circular_modes(angles, bins=72, sigma=2.0, prominence_fraction=0.08, min_distance=5):
    angles = (np.asarray(angles) + np.pi) % (2*np.pi) - np.pi
    hist, edges = np.histogram(angles, bins=bins, range=(-np.pi, np.pi), density=True)
    centers = (edges[:-1] + edges[1:]) / 2
    smooth = gaussian_filter1d(hist.astype(float), sigma=sigma, mode="wrap")
    extended = np.tile(smooth, 3)
    prominence = max(prominence_fraction * (smooth.max() - smooth.min()), 1e-12)
    peaks, _ = find_peaks(extended, prominence=prominence, distance=min_distance)
    peaks = peaks[(peaks >= bins) & (peaks < 2*bins)] - bins
    unique = []
    for peak in peaks:
        if all(min(abs(peak-q), bins-abs(peak-q)) >= min_distance for q in unique):
            unique.append(int(peak))
    return centers, hist, smooth, np.asarray(unique, dtype=int)

MODE_DETAILS = {}
rows = []
for j in range(THETA_WRAPPED.shape[1]):
    centers, hist, smooth, peaks = circular_modes(THETA_WRAPPED[:, j])
    MODE_DETAILS[j] = dict(centers=centers, hist=hist, smooth=smooth, peaks=peaks)
    rows.append({
        "theta_index": j,
        "estimated_modes": max(1, len(peaks)),
        "peak_angles": centers[peaks].tolist() if len(peaks) else [],
    })

mode_table = pd.DataFrame(rows)
display(mode_table)
mode_table.to_json(TABLE_DIR / "angular_modes.json", orient="records", indent=2)

multimodal_indices = mode_table.loc[mode_table["estimated_modes"] > 1, "theta_index"].astype(int).tolist()
n_multimodal = len(multimodal_indices)
fraction_multimodal = n_multimodal / len(mode_table)

fig, ax = plt.subplots()
ax.bar(mode_table["theta_index"], mode_table["estimated_modes"])
for _, row in mode_table.iterrows():
    if row["estimated_modes"] > 1:
        ax.annotate(
            f"θ{int(row['theta_index'])}",
            xy=(row["theta_index"], row["estimated_modes"]),
            xytext=(0, 5), textcoords="offset points",
            ha="center", va="bottom", fontsize=9, fontweight="bold",
        )

ax.set_title("Quais componentes possuem uma ou duas regiões preferenciais?")
ax.set_xlabel("Índice do parâmetro")
ax.set_ylabel("Número estimado de modos")
ax.set_xticks(np.arange(THETA_WRAPPED.shape[1]))
ax.set_yticks(np.arange(1, int(mode_table["estimated_modes"].max()) + 1))
ax.grid(axis="y", alpha=0.25)
info_box(ax, [
    f"Unimodais: {len(mode_table)-n_multimodal}/{len(mode_table)}",
    f"Multimodais: {n_multimodal}/{len(mode_table)}",
    f"Fração multimodal: {fraction_multimodal:.1%}",
    "Índices: " + ", ".join(f"θ{i}" for i in multimodal_indices),
])
short_takeaway(ax, "Valor 2 significa duas regiões finais preferenciais para a componente, não dois mínimos físicos.")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "02_multimodalidade_INFORMATIVO.png", dpi=300, bbox_inches="tight")
plt.show()

## Exemplos de distribuição angular

### Pergunta respondida

**Onde uma componente termina ao longo das execuções?**

As barras são a densidade observada; a linha é a versão suavizada.  
A densidade próxima de \(-\pi\) pode ser a continuação de uma região próxima de \(+\pi\).

- \(\theta_1\): exemplo esperado de região ampla, porém única;
- \(\theta_{11}\): exemplo esperado de duas regiões.

In [ ]:
def plot_angular_distribution(j, filename):
    d = MODE_DETAILS[j]
    centers, hist, smooth, peaks = d["centers"], d["hist"], d["smooth"], d["peaks"]
    width = 2*np.pi / len(centers)
    n_modes = max(1, len(peaks))
    peak_values = [float(centers[p]) for p in peaks]

    fig, ax = plt.subplots()
    ax.bar(centers, hist, width=width, alpha=0.65, label="Frequência observada")
    ax.plot(centers, smooth, linewidth=2.2, label="Densidade circular suavizada")

    for number, p in enumerate(peaks, start=1):
        angle_rad = float(centers[p])
        angle_deg = float(np.degrees(angle_rad))
        ax.axvline(angle_rad, linestyle="--", alpha=0.85)
        ax.annotate(
            f"Modo {number}\n{angle_rad:.2f} rad\n{angle_deg:.0f}°",
            xy=(angle_rad, smooth[p]), xytext=(10, 18), textcoords="offset points",
            ha="left", arrowprops={"arrowstyle": "->"}, fontsize=9,
        )

    variance_j = float(CIRCULAR_VARIANCE[j])
    resultant_j = float(RESULTANT_LENGTH[j])
    ax.set_title(f"Onde θ{j} termina nas {len(THETA_WRAPPED):,} execuções?")
    ax.set_xlabel(f"θ{j} no círculo [−π, π)")
    ax.set_ylabel("Densidade")
    ax.set_xlim(-np.pi, np.pi)
    ax.set_xticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
    ax.set_xticklabels([r"$-\pi$", r"$-\pi/2$", "0", r"$\pi/2$", r"$\pi$"])
    ax.legend(loc="upper left")
    ax.grid(alpha=0.2)

    lines = [
        f"Modos estimados = {n_modes}",
        f"Variância circular = {variance_j:.3f}",
        f"Concentração R = {resultant_j:.3f}",
    ]
    if n_modes > 1 and len(peak_values) >= 2:
        separation = abs(np.arctan2(
            np.sin(peak_values[1] - peak_values[0]),
            np.cos(peak_values[1] - peak_values[0]),
        ))
        lines.append(f"Separação circular ≈ {separation:.2f} rad")
    info_box(ax, lines)

    if np.any(hist[:4] > 0) and np.any(hist[-4:] > 0):
        ax.text(
            0.5, 0.04,
            "As duas bordas são contínuas: −π e +π representam o mesmo ponto angular.",
            transform=ax.transAxes, ha="center", fontsize=9,
        )
    short_takeaway(ax, "O gráfico mede a frequência dos valores finais; não mostra a trajetória do otimizador.")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()

if UNIMODAL_EXAMPLE_INDEX < THETA_WRAPPED.shape[1]:
    plot_angular_distribution(UNIMODAL_EXAMPLE_INDEX, "03_theta_unimodal_INFORMATIVO.png")
if BIMODAL_EXAMPLE_INDEX < THETA_WRAPPED.shape[1]:
    plot_angular_distribution(BIMODAL_EXAMPLE_INDEX, "04_theta_bimodal_INFORMATIVO.png")

# 5. PCA: dimensão necessária

### Pergunta respondida

**Quantos padrões lineares combinados são necessários para representar a diversidade?**

Cada ângulo é representado por:

\[
\theta_j\longrightarrow(\cos\theta_j,\sin\theta_j).
\]

O PCA trabalha no vetor de 60 coordenadas e calcula

\[
C(m)=\frac{\sum_{i=1}^{m}\lambda_i}{\sum_i\lambda_i}.
\]

O número obtido não é o número de mínimos ou soluções; ele mede a compressibilidade linear da nuvem angular.

In [ ]:
ANGULAR_EMBEDDING = np.concatenate([np.cos(THETA_WRAPPED), np.sin(THETA_WRAPPED)], axis=1)
PCA_FULL = PCA()
PCA_COORDINATES_FULL = PCA_FULL.fit_transform(ANGULAR_EMBEDDING)
explained = PCA_FULL.explained_variance_ratio_
cumulative = np.cumsum(explained)
n90 = int(np.searchsorted(cumulative, 0.90) + 1)
n95 = int(np.searchsorted(cumulative, 0.95) + 1)
pc12 = float(cumulative[1] if len(cumulative) > 1 else cumulative[0])

pca_table = pd.DataFrame({
    "component": np.arange(1, len(explained)+1),
    "explained_variance_ratio": explained,
    "cumulative_variance": cumulative,
})
pca_table.to_csv(TABLE_DIR / "pca_variance.csv", index=False)

fig, ax = plt.subplots()
ax.plot(pca_table["component"], pca_table["cumulative_variance"], marker="o", markersize=3)
ax.axhline(0.90, linestyle="--", label="Meta de 90%")
ax.axhline(0.95, linestyle=":", label="Meta de 95%")
ax.axvline(n90, linestyle="--", alpha=0.75)
ax.axvline(n95, linestyle=":", alpha=0.75)
ax.scatter([n90, n95], [cumulative[n90-1], cumulative[n95-1]], s=80)
ax.annotate(f"{n90} componentes\npreservam 90%", xy=(n90, cumulative[n90-1]), xytext=(-95, 22), textcoords="offset points", arrowprops={"arrowstyle": "->"}, fontsize=10)
ax.annotate(f"{n95} componentes\npreservam 95%", xy=(n95, cumulative[n95-1]), xytext=(15, -45), textcoords="offset points", arrowprops={"arrowstyle": "->"}, fontsize=10)
ax.set_title("Quantas direções são necessárias para representar a diversidade angular?")
ax.set_xlabel("Número de componentes principais mantidas")
ax.set_ylabel("Variância angular explicada acumulada")
ax.set_ylim(0, 1.02)
ax.legend(loc="lower right")
ax.grid(alpha=0.25)
info_box(ax, [
    f"Espaço de entrada: {ANGULAR_EMBEDDING.shape[1]} coordenadas",
    f"PC1 + PC2 = {pc12:.1%}",
    f"90% → {n90} componentes",
    f"95% → {n95} componentes",
])
short_takeaway(ax, "A subida gradual mostra que a diversidade está espalhada por muitas combinações de parâmetros.")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "05_pca_variancia_acumulada_INFORMATIVO.png", dpi=300, bbox_inches="tight")
plt.show()

# 6. Seleção do número de clusters angulares

### Pergunta respondida

**Os vetores completos formam famílias globais claramente separadas?**

Para cada \(k\), o KMeans particiona o espaço seno–cosseno. A qualidade é medida pelo silhouette:

\[
s=\frac{b-a}{\max(a,b)}.
\]

- próximo de 1: grupos bem separados;
- próximo de 0: grupos sobrepostos;
- negativo: muitos pontos mal atribuídos.

O maior valor entre os testados não é necessariamente um bom valor absoluto.

In [ ]:
silhouette_rows = []
models = {}
for k in K_RANGE:
    model = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    labels = model.fit_predict(ANGULAR_EMBEDDING)
    score = silhouette_score(
        ANGULAR_EMBEDDING, labels,
        sample_size=min(SILHOUETTE_SAMPLE_SIZE, len(labels)),
        random_state=RANDOM_STATE,
    )
    silhouette_rows.append({"k": k, "silhouette": score})
    models[k] = (model, labels)

silhouette_table = pd.DataFrame(silhouette_rows)
BEST_K = int(silhouette_table.loc[silhouette_table["silhouette"].idxmax(), "k"])
BEST_SILHOUETTE = float(silhouette_table["silhouette"].max())
BEST_MODEL, CLUSTER_LABELS = models[BEST_K]
silhouette_table.to_csv(TABLE_DIR / "kmeans_silhouette.csv", index=False)

fig, ax = plt.subplots()
ax.plot(silhouette_table["k"], silhouette_table["silhouette"], marker="o")
for _, row in silhouette_table.iterrows():
    ax.annotate(f"{row['silhouette']:.3f}", xy=(row["k"], row["silhouette"]), xytext=(0, 7), textcoords="offset points", ha="center", fontsize=9)
ax.scatter([BEST_K], [BEST_SILHOUETTE], s=120, label=f"Maior valor: k={BEST_K}")
ax.set_title("Existe um número natural de famílias angulares?")
ax.set_xlabel("Número de clusters impostos pelo KMeans")
ax.set_ylabel("Silhouette médio")
ax.set_xticks(list(K_RANGE))
ax.legend()
ax.grid(alpha=0.25)
quality_text = "separação fraca / grande sobreposição" if BEST_SILHOUETTE < 0.10 else "separação moderada" if BEST_SILHOUETTE < 0.50 else "separação forte"
info_box(ax, [
    f"Melhor k = {BEST_K}",
    f"Silhouette = {BEST_SILHOUETTE:.3f}",
    f"Interpretação: {quality_text}",
    "Próximo de 0 → grupos sobrepostos",
])
short_takeaway(ax, "O maior valor é apenas o melhor entre os k testados; um silhouette baixo não confirma clusters naturais.")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "06_selecao_k_INFORMATIVO.png", dpi=300, bbox_inches="tight")
plt.show()

## Projeção PCA colorida pelos clusters

### Pergunta respondida

**A melhor partição aparece visualmente como ilhas separadas?**

A figura é apenas uma projeção em duas dimensões. A conclusão principal vem do silhouette calculado no espaço completo.

In [ ]:
PCA_2D = PCA(n_components=2)
PCA_2D_COORDS = PCA_2D.fit_transform(ANGULAR_EMBEDDING)
cluster_sizes = pd.Series(CLUSTER_LABELS).value_counts().sort_index()

fig, ax = plt.subplots()
scatter = ax.scatter(PCA_2D_COORDS[:, 0], PCA_2D_COORDS[:, 1], c=CLUSTER_LABELS, s=10, alpha=0.42)
for cluster_id in sorted(np.unique(CLUSTER_LABELS)):
    mask = CLUSTER_LABELS == cluster_id
    center = PCA_2D_COORDS[mask].mean(axis=0)
    ax.scatter(center[0], center[1], marker="X", s=170, edgecolors="black", linewidths=1.0)
    ax.annotate(f"Cluster {cluster_id}\nn={mask.sum():,}", xy=center, xytext=(8, 8), textcoords="offset points", fontsize=9, fontweight="bold")

pc1 = float(PCA_2D.explained_variance_ratio_[0])
pc2 = float(PCA_2D.explained_variance_ratio_[1])
ax.set_title("A partição do KMeans forma ilhas visualmente separadas?")
ax.set_xlabel(f"PC1 ({pc1*100:.2f}% da variância)")
ax.set_ylabel(f"PC2 ({pc2*100:.2f}% da variância)")
ax.grid(alpha=0.2)
legend = ax.legend(*scatter.legend_elements(), title="Rótulo KMeans")
ax.add_artist(legend)
info_box(ax, [
    f"k = {BEST_K}",
    f"Silhouette = {BEST_SILHOUETTE:.3f}",
    f"PC1 + PC2 = {(pc1+pc2):.1%}",
    "Centros marcados por X",
])
short_takeaway(ax, "Forte sobreposição visual + silhouette próximo de zero indicam uma nuvem contínua dividida artificialmente.")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "07_pca_clusters_INFORMATIVO.png", dpi=300, bbox_inches="tight")
plt.show()

# 7. Distribuição do erro energético

### Pergunta respondida

**Todas as execuções possuem a mesma qualidade?**

\[
\Delta E=\left|E_{\mathrm{VQE}}-E_{\mathrm{exato}}\right|.
\]

- próximo de zero: quase ótimo;
- segundo pico: família subótima recorrente;
- cauda: execuções de baixa qualidade.

In [ ]:
energy_error = analysis_df["absolute_energy_error"].to_numpy()
fraction_1e3 = float((energy_error < 1e-3).mean())
fraction_1e2 = float((energy_error < 1e-2).mean())
mean_error = float(energy_error.mean())
median_error = float(np.median(energy_error))
max_error = float(energy_error.max())

fig, ax = plt.subplots()
ax.hist(energy_error, bins=70, density=False, alpha=0.75)
ax.axvline(1e-3, linestyle="--", label=f"Erro < 10⁻³: {fraction_1e3:.1%}")
ax.axvline(1e-2, linestyle=":", label=f"Erro < 10⁻²: {fraction_1e2:.1%}")
ax.axvline(mean_error, linestyle="-.", label=f"Média = {mean_error:.4f}")
ax.axvline(median_error, linestyle=(0, (5, 3)), label=f"Mediana = {median_error:.4f}")
ax.set_title("As execuções possuem a mesma qualidade energética?")
ax.set_xlabel(r"Erro absoluto  $\Delta E=|E_{\mathrm{VQE}}-E_{\mathrm{exato}}|$")
ax.set_ylabel("Número de execuções")
ax.legend()
ax.grid(alpha=0.2)
info_box(ax, [
    f"E exata = {EXACT_ENERGY:.6f}",
    f"Erro médio = {mean_error:.4f}",
    f"Erro mediano = {median_error:.4f}",
    f"Erro máximo = {max_error:.4f}",
    f"Quase ótimas (<10⁻³): {fraction_1e3:.1%}",
    f"Aceitáveis (<10⁻²): {fraction_1e2:.1%}",
])
short_takeaway(ax, "O banco mistura soluções quase ótimas, uma família subótima recorrente e uma pequena cauda de casos ruins.")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "08_erro_energetico_INFORMATIVO.png", dpi=300, bbox_inches="tight")
plt.show()

# 8. Frequência dos bitstrings

### Pergunta respondida

**Quais soluções clássicas aparecem com maior frequência?**

Frequência não é sinônimo de qualidade. Um bitstring frequente pode ser subótimo; por isso, este gráfico deve ser apresentado junto com a energia por bitstring.

In [ ]:
valid_bits_df = analysis_df.dropna(subset=["bitstring"]).copy()
if valid_bits_df.empty:
    print("[AVISO] Nenhum bitstring válido.")
else:
    bit_counts = valid_bits_df["bitstring"].value_counts().rename_axis("bitstring").reset_index(name="count")
    bit_counts["fraction"] = bit_counts["count"] / len(valid_bits_df)
    bit_counts.to_csv(TABLE_DIR / "bitstring_frequencies.csv", index=False)

    top = bit_counts.head(TOP_BITSTRINGS).iloc[::-1]
    fig, ax = plt.subplots()
    bars = ax.barh(top["bitstring"], top["count"])
    labels = [f"{count:,}  ({fraction:.1%})" for count, fraction in zip(top["count"], top["fraction"])]
    label_horizontal_bars(ax, bars, labels)
    ax.set_title("Quais soluções clássicas aparecem com maior frequência?")
    ax.set_xlabel("Número de execuções")
    ax.set_ylabel("Bitstring dominante")
    ax.grid(axis="x", alpha=0.25)

    if EXACT_BITSTRING in top["bitstring"].values:
        row = top[top["bitstring"] == EXACT_BITSTRING].iloc[0]
        ax.annotate("Solução exata", xy=(row["count"], row["bitstring"]), xytext=(-95, 0), textcoords="offset points", va="center", fontweight="bold", arrowprops={"arrowstyle": "->"})

    top_bit = str(bit_counts.iloc[0]["bitstring"])
    top_fraction = float(bit_counts.iloc[0]["fraction"])
    exact_fraction = float((valid_bits_df["bitstring"] == EXACT_BITSTRING).mean())
    info_box(ax, [
        f"Total válido = {len(valid_bits_df):,}",
        f"Mais frequente = {top_bit}",
        f"Fração do mais frequente = {top_fraction:.1%}",
        f"Fração da solução exata = {exact_fraction:.1%}",
    ])
    short_takeaway(ax, "Frequência não é sinônimo de qualidade: compare com a energia associada a cada bitstring.")
    ax.margins(x=0.18)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "09_frequencia_bitstrings_INFORMATIVO.png", dpi=300, bbox_inches="tight")
    plt.show()
    display(bit_counts.head(20))

# 9. Energia associada a cada bitstring

### Pergunta respondida

**O mesmo bitstring aparece sempre com a mesma energia? Quais bitstrings concentram as menores energias?**

O estado VQE pode ser

\[
|\psi(\theta)\rangle=\sum_x c_x(\theta)|x\rangle,
\]

e sua energia é

\[
E(\theta)=\langle\psi(\theta)|H|\psi(\theta)\rangle.
\]

Por isso, o mesmo bitstring dominante pode aparecer com energias diferentes: as probabilidades dos demais estados também mudam.

In [ ]:
if valid_bits_df.empty:
    print("[AVISO] Gráfico ignorado.")
else:
    frequent = valid_bits_df["bitstring"].value_counts().head(TOP_BITSTRINGS).index.tolist()
    subset = valid_bits_df[valid_bits_df["bitstring"].isin(frequent)]
    order_summary = subset.groupby("bitstring")["energy"].agg(["mean", "median", "count", "std"]).sort_values(["mean", "count"], ascending=[True, False])
    order = order_summary.index.tolist()
    groups = [subset.loc[subset["bitstring"] == bit, "energy"].to_numpy() for bit in order]
    rng = np.random.default_rng(RANDOM_STATE)

    fig, ax = plt.subplots(figsize=(15, 8))
    ax.boxplot(groups, tick_labels=order, showfliers=False, widths=0.55)
    for position, values in enumerate(groups, start=1):
        values_plot = rng.choice(values, 1000, replace=False) if len(values) > 1000 else values
        jitter = rng.uniform(-0.18, 0.18, len(values_plot))
        ax.scatter(position + jitter, values_plot, s=8, alpha=0.22)
        mean_value = float(np.mean(values))
        ax.scatter(position, mean_value, marker="D", s=55, edgecolors="black", linewidths=0.8)
        ax.annotate(f"n={len(values):,}\nμ={mean_value:.3f}", xy=(position, mean_value), xytext=(0, 10), textcoords="offset points", ha="center", fontsize=8)

    ax.axhline(EXACT_ENERGY, linestyle="--", label=f"Energia exata = {EXACT_ENERGY:.6f}")
    ax.set_title("Qual energia está associada a cada bitstring dominante?")
    ax.set_xlabel("Bitstring dominante")
    ax.set_ylabel("Energia final do VQE")
    ax.tick_params(axis="x", rotation=55)
    ax.legend()
    ax.grid(axis="y", alpha=0.25)

    best_mean_bit = order_summary["mean"].idxmin()
    best_mean_energy = float(order_summary.loc[best_mean_bit, "mean"])
    info_box(ax, [
        "Pontos = execuções individuais",
        "Caixa = distribuição da energia",
        "Losango = energia média",
        f"Melhor média: {best_mean_bit}",
        f"μ = {best_mean_energy:.6f}",
    ])
    short_takeaway(ax, "O mesmo bitstring pode aparecer com energias diferentes porque o estado completo contém outras amplitudes.")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "10_energia_por_bitstring_INFORMATIVO.png", dpi=300, bbox_inches="tight")
    plt.show()

    bit_energy_summary = valid_bits_df.groupby("bitstring")["energy"].agg(
        count="size", mean_energy="mean", median_energy="median",
        min_energy="min", max_energy="max", std_energy="std"
    ).sort_values(["mean_energy", "count"], ascending=[True, False])
    bit_energy_summary.to_csv(TABLE_DIR / "energy_by_bitstring.csv")
    display(bit_energy_summary.head(20))

# 10. Mapa bitstring × faixa de energia

### Pergunta respondida

**Quais famílias de bitstrings ocupam cada região energética?**

As linhas são bitstrings, as colunas são faixas de energia e a intensidade representa o número de execuções.  
Este é o gráfico mais direto para mostrar a família quase ótima e a família subótima recorrente.

In [ ]:
def energy_bin(values, tolerance):
    return np.rint(np.asarray(values, dtype=float) / tolerance) * tolerance

if valid_bits_df.empty:
    print("[AVISO] Gráfico ignorado.")
else:
    heat = valid_bits_df.copy()
    heat["energy_bin"] = energy_bin(heat["energy"], ENERGY_BIN_TOL)
    frequent = heat["bitstring"].value_counts().head(TOP_BITSTRINGS).index
    heat = heat[heat["bitstring"].isin(frequent)]
    matrix = pd.crosstab(heat["bitstring"], heat["energy_bin"])
    row_order = heat.groupby("bitstring")["energy"].mean().sort_values().index
    matrix = matrix.reindex(row_order)
    matrix = matrix.loc[:, matrix.sum(axis=0) > 0]

    fig, ax = plt.subplots(figsize=(max(13, 0.36*matrix.shape[1]), 8))
    image = ax.imshow(matrix.to_numpy(), aspect="auto", interpolation="nearest")
    nonzero = matrix.to_numpy()[matrix.to_numpy() > 0]
    threshold = np.quantile(nonzero, 0.80) if len(nonzero) else np.inf
    for r in range(matrix.shape[0]):
        for c in range(matrix.shape[1]):
            value = int(matrix.iloc[r, c])
            if value >= threshold:
                ax.text(c, r, str(value), ha="center", va="center", fontsize=8, fontweight="bold")

    ax.set_title("Em quais faixas de energia cada bitstring se concentra?")
    ax.set_xlabel(f"Centro da faixa de energia  (largura = {ENERGY_BIN_TOL:g})")
    ax.set_ylabel("Bitstring dominante")
    ylabels = [f"★ {bit}" if bit == EXACT_BITSTRING else bit for bit in matrix.index]
    ax.set_yticks(np.arange(len(matrix.index)))
    ax.set_yticklabels(ylabels)
    positions = np.arange(len(matrix.columns))
    step = max(1, int(np.ceil(len(positions)/18)))
    shown = positions[::step]
    ax.set_xticks(shown)
    ax.set_xticklabels([f"{matrix.columns[i]:.3f}" for i in shown], rotation=60, ha="right")
    cbar = fig.colorbar(image, ax=ax)
    cbar.set_label("Número de execuções")

    pos = np.unravel_index(np.argmax(matrix.to_numpy()), matrix.shape)
    largest_bit = matrix.index[pos[0]]
    largest_energy = matrix.columns[pos[1]]
    largest_count = int(matrix.iloc[pos])
    info_box(ax, [
        "★ = bitstring exato",
        f"Maior concentração: {largest_bit}",
        f"E ≈ {largest_energy:.3f}",
        f"n = {largest_count:,}",
        "Números = células mais densas",
    ])
    short_takeaway(ax, "Bandas horizontais mostram como um mesmo bitstring se distribui em várias faixas de energia.")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "11_heatmap_bitstring_energia_INFORMATIVO.png", dpi=300, bbox_inches="tight")
    plt.show()
    matrix.to_csv(TABLE_DIR / "bitstring_energy_matrix.csv")

# 11. Degenerescência e redundância

## Degenerescência exata

Estados distintos possuem o mesmo autovalor:

\[
H|\psi_a\rangle=E|\psi_a\rangle,
\qquad
H|\psi_b\rangle=E|\psi_b\rangle,
\qquad
|\psi_a\rangle\neq|\psi_b\rangle.
\]

Isso exige uma verificação exata do espectro.

## Quase-degenerescência operacional

Bitstrings diferentes aparecem na mesma faixa de energia:

\[
|E_a-E_b|<\varepsilon.
\]

É dependente da tolerância \(\varepsilon\) e não prova degenerescência exata.

## Redundância paramétrica

Vetores \(\theta\) diferentes produzem aproximadamente o mesmo bitstring e a mesma energia.  
Essa é a situação mais claramente sugerida pelo banco atual.

## Bitstrings distintos por faixa de energia

### Pergunta respondida

**Quantos bitstrings diferentes aparecem dentro de cada faixa energética?**

Valores acima de 1 apontam faixas que merecem uma investigação exata de quase-degenerescência.

In [ ]:
if valid_bits_df.empty:
    print("[AVISO] Gráfico ignorado.")
else:
    deg = valid_bits_df.copy()
    deg["energy_bin"] = energy_bin(deg["energy"], ENERGY_BIN_TOL)
    operational = deg.groupby("energy_bin").agg(
        distinct_bitstrings=("bitstring", "nunique"),
        executions=("bitstring", "size"),
    ).reset_index().sort_values("energy_bin")

    sizes = 25 + 220 * operational["executions"] / operational["executions"].max()
    fig, ax = plt.subplots()
    ax.scatter(operational["energy_bin"], operational["distinct_bitstrings"], s=sizes, alpha=0.65)
    ax.axhline(1, linestyle="--", label="Apenas um bitstring na faixa")

    top_candidates = operational[operational["distinct_bitstrings"] > 1].sort_values(["distinct_bitstrings", "executions"], ascending=False).head(5)
    for _, row in top_candidates.iterrows():
        ax.annotate(
            f"E≈{row['energy_bin']:.3f}\n{int(row['distinct_bitstrings'])} bits\nn={int(row['executions'])}",
            xy=(row["energy_bin"], row["distinct_bitstrings"]), xytext=(8, 8), textcoords="offset points", fontsize=8, arrowprops={"arrowstyle": "->"},
        )

    candidate_bins = int((operational["distinct_bitstrings"] > 1).sum())
    ax.set_title("Quais faixas contêm bitstrings diferentes com energias próximas?")
    ax.set_xlabel("Centro da faixa de energia")
    ax.set_ylabel("Número de bitstrings dominantes distintos")
    ax.legend()
    ax.grid(alpha=0.25)
    info_box(ax, [
        f"Tolerância = {ENERGY_BIN_TOL:g}",
        f"Faixas analisadas = {len(operational)}",
        f"Candidatas (>1 bitstring) = {candidate_bins}",
        "Tamanho da bolha = nº de execuções",
        "Não prova degenerescência exata",
    ])
    short_takeaway(ax, "Mais de um bitstring na mesma faixa indica quase-degenerescência operacional dependente da tolerância.")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "12_quase_degenerescencia_INFORMATIVO.png", dpi=300, bbox_inches="tight")
    plt.show()
    operational.to_csv(TABLE_DIR / "near_degeneracy_operational.csv", index=False)
    display(operational[operational["distinct_bitstrings"] > 1].sort_values(["distinct_bitstrings", "executions"], ascending=False).head(20))

# 12. Classes operacionais e dispersão angular

### Pergunta respondida

**Quantos vetores diferentes produzem aproximadamente o mesmo bitstring e a mesma energia?**

Uma classe é

\[
(\text{bitstring dominante},\text{faixa de energia}).
\]

A dispersão interna é a distância toroidal até a média circular da classe:

\[
d_{\mathbb T}
=
\sqrt{\sum_j
\operatorname{atan2}\left[
\sin(\theta_j-\bar\theta_j),
\cos(\theta_j-\bar\theta_j)
\right]^2}.
\]

Classe grande e angularmente espalhada é evidência de redundância paramétrica.

In [ ]:
def toroidal_distance(theta_matrix, reference):
    delta = np.arctan2(np.sin(theta_matrix-reference), np.cos(theta_matrix-reference))
    return np.linalg.norm(delta, axis=1)

if valid_bits_df.empty:
    print("[AVISO] Análise ignorada.")
else:
    classes = valid_bits_df.copy()
    classes["energy_bin"] = energy_bin(classes["energy"], ENERGY_BIN_TOL)
    decimals = max(0, int(round(-math.log10(ENERGY_BIN_TOL))))
    classes["equivalence_class"] = classes["bitstring"].astype(str) + "__" + classes["energy_bin"].map(lambda x: f"{x:.{decimals}f}")

    rows = []
    for name, group in classes.groupby("equivalence_class"):
        positions = group.index.to_numpy()
        theta_group = THETA_WRAPPED[positions]
        mean_angle = np.arctan2(np.mean(np.sin(theta_group), axis=0), np.mean(np.cos(theta_group), axis=0))
        distances = toroidal_distance(theta_group, mean_angle)
        rows.append({
            "equivalence_class": name,
            "bitstring": group["bitstring"].iloc[0],
            "energy_bin": group["energy_bin"].iloc[0],
            "size": len(group),
            "mean_angular_spread": distances.mean(),
            "max_angular_spread": distances.max(),
            "mean_energy": group["energy"].mean(),
        })

    equivalence_table = pd.DataFrame(rows).sort_values("size", ascending=False).reset_index(drop=True)
    equivalence_table["fraction"] = equivalence_table["size"] / len(valid_bits_df)
    equivalence_table.to_csv(TABLE_DIR / "equivalence_classes.csv", index=False)
    display(equivalence_table.head(20))

    top = equivalence_table.head(TOP_EQUIVALENCE_CLASSES).iloc[::-1]
    fig, ax = plt.subplots(figsize=(13, 8))
    bars = ax.barh(top["equivalence_class"], top["size"])
    labels = [f"{size:,} ({fraction:.1%})" for size, fraction in zip(top["size"], top["fraction"])]
    label_horizontal_bars(ax, bars, labels)
    ax.set_title("Quais classes operacionais reúnem mais execuções?")
    ax.set_xlabel("Número de execuções")
    ax.set_ylabel("Bitstring + faixa aproximada de energia")
    ax.grid(axis="x", alpha=0.25)
    ax.margins(x=0.20)

    largest = equivalence_table.iloc[0]
    info_box(ax, [
        f"Classes encontradas = {len(equivalence_table)}",
        f"Maior classe = {largest['equivalence_class']}",
        f"Tamanho = {int(largest['size']):,}",
        f"Fração = {largest['fraction']:.1%}",
        f"Dispersão angular = {largest['mean_angular_spread']:.2f}",
    ])
    short_takeaway(ax, "Classes grandes agrupam muitas execuções com o mesmo bitstring dominante e energia aproximadamente igual.")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "13_classes_operacionais_INFORMATIVO.png", dpi=300, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots()
    ax.scatter(equivalence_table["size"], equivalence_table["mean_angular_spread"], s=30, alpha=0.65)
    for _, row in equivalence_table.head(5).iterrows():
        ax.annotate(row["equivalence_class"], xy=(row["size"], row["mean_angular_spread"]), xytext=(7, 6), textcoords="offset points", fontsize=8)
    ax.set_title("Classes grandes também possuem θ angularmente diferentes?")
    ax.set_xlabel("Tamanho da classe")
    ax.set_ylabel("Dispersão angular média interna")
    ax.grid(alpha=0.25)
    info_box(ax, [
        "Cada ponto = uma classe operacional",
        "Direita = classe mais frequente",
        "Acima = maior diversidade de θ",
        "Grande + alta = redundância paramétrica",
    ])
    short_takeaway(ax, "Uma classe grande e dispersa mostra que muitos vetores θ diferentes produzem aproximadamente o mesmo resultado.")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "14_classe_vs_dispersao_INFORMATIVO.png", dpi=300, bbox_inches="tight")
    plt.show()

# 13. Erro energético × distância angular

### Pergunta respondida

**Vetores mais afastados do centro angular são necessariamente piores?**

Se não houver tendência clara, a distância geométrica entre parâmetros não é suficiente para prever a qualidade energética.

In [ ]:
distance_to_mean = toroidal_distance(THETA_WRAPPED, CIRCULAR_MEAN)
analysis_df["distance_to_circular_mean"] = distance_to_mean
correlation = float(np.corrcoef(distance_to_mean, analysis_df["absolute_energy_error"])[0, 1])

distance_bins = pd.qcut(distance_to_mean, q=12, duplicates="drop")
trend = pd.DataFrame({
    "distance": distance_to_mean,
    "error": analysis_df["absolute_energy_error"].to_numpy(),
    "distance_bin": distance_bins,
}).groupby("distance_bin", observed=True).agg(
    distance_median=("distance", "median"),
    error_median=("error", "median"),
    count=("error", "size"),
).reset_index()

fig, ax = plt.subplots()
ax.scatter(distance_to_mean, analysis_df["absolute_energy_error"], s=10, alpha=0.25, label="Execuções")
ax.plot(trend["distance_median"], trend["error_median"], marker="o", linewidth=2.2, label="Mediana por faixa")
ax.set_title("Vetores mais distantes angularmente possuem maior erro?")
ax.set_xlabel("Distância toroidal à média circular")
ax.set_ylabel(r"Erro energético absoluto  $\Delta E$")
ax.legend()
ax.grid(alpha=0.25)
relation_text = "relação linear muito fraca" if abs(correlation) < 0.20 else "relação linear moderada" if abs(correlation) < 0.60 else "relação linear forte"
info_box(ax, [
    f"Correlação = {correlation:.3f}",
    f"Interpretação: {relation_text}",
    "Linha = mediana em 12 faixas",
    "Distância angular ≠ qualidade física",
])
short_takeaway(ax, "Se a linha permanece quase horizontal, a distância geométrica entre θ não prevê a qualidade energética.")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "15_erro_vs_distancia_INFORMATIVO.png", dpi=300, bbox_inches="tight")
plt.show()
print("Correlação:", correlation)

# 14. Resumo automático para apresentação

A leitura recomendada é:

1. A diversidade angular existe, mas é desigual entre as componentes.
2. Alguns parâmetros possuem dois modos marginais, sem que isso prove dois mínimos físicos.
3. O PCA indica diversidade distribuída em muitas direções.
4. O KMeans só deve ser interpretado como forte se o silhouette for claramente superior a zero.
5. A energia separa soluções quase ótimas e subótimas.
6. Bitstrings e faixas energéticas revelam famílias funcionais.
7. Classes grandes e angularmente dispersas indicam redundância paramétrica.
8. Quase-degenerescência operacional não é degenerescência exata.

In [ ]:
multimodal_indices = mode_table.loc[
    mode_table["estimated_modes"] > 1, "theta_index"
].tolist()

best_bit = (
    valid_bits_df["bitstring"].value_counts().index[0]
    if not valid_bits_df.empty else None
)

print("="*72)
print("RESUMO PARA APRESENTAÇÃO")
print("="*72)
print(f"Execuções: {len(analysis_df):,}")
print(f"Parâmetros: {THETA_WRAPPED.shape[1]}")
print(f"Variância circular média: {CIRCULAR_VARIANCE.mean():.4f}")
print(f"Componentes multimodais: {len(multimodal_indices)} -> {multimodal_indices}")
print(f"PCA: {n90} componentes para 90%")
print(f"PCA: {n95} componentes para 95%")
print(f"KMeans: k={BEST_K}, silhouette={BEST_SILHOUETTE:.4f}")
print(f"Erro < 1e-3: {(energy_error < 1e-3).mean():.2%}")
print(f"Erro < 1e-2: {(energy_error < 1e-2).mean():.2%}")
print(f"Bitstring mais frequente: {best_bit}")
print(f"Fração com bitstring exato: {(analysis_df['bitstring'] == EXACT_BITSTRING).mean():.2%}")

if "equivalence_table" in globals() and not equivalence_table.empty:
    print("\nMaiores classes:")
    print(equivalence_table[
        ["equivalence_class", "size", "mean_angular_spread"]
    ].head(5).to_string(index=False))

print("\nCONCLUSÃO:")
print(
    "O banco contém diversidade paramétrica angular real, mas ela não "
    "necessariamente corresponde a muitas soluções físicas distintas. "
    "A energia e os bitstrings revelam famílias quase ótimas e subótimas, "
    "enquanto classes grandes e dispersas mostram redundância paramétrica."
)

# Conclusão científica

Os gráficos agora foram desenhados para sustentar uma narrativa visual direta:

1. **Variância circular:** a diversidade existe, mas não é distribuída igualmente entre os parâmetros.
2. **Multimodalidade:** algumas componentes possuem duas regiões finais preferenciais, sem que isso prove dois mínimos físicos.
3. **PCA:** a diversidade angular é pouco compressível por poucas direções lineares.
4. **KMeans:** a melhor divisão matemática pode continuar sendo uma divisão fraca e artificial.
5. **Erro energético:** o banco mistura resultados quase ótimos, subótimos e uma pequena cauda ruim.
6. **Bitstring × energia:** a estrutura funcional é mais clara quando solução clássica e energia são analisadas conjuntamente.
7. **Classes operacionais:** muitos vetores θ angularmente diferentes podem gerar aproximadamente o mesmo resultado.
8. **Quase-degenerescência:** bitstrings diferentes em uma mesma faixa energética são candidatos operacionais, não prova de degenerescência exata.

> **Mensagem central:** os 10.000 vetores não devem ser tratados como 10.000 respostas ótimas independentes. O banco representa uma distribuição estruturada de parametrizações, com redundância, diferentes níveis de qualidade e poucas famílias funcionais dominantes.